# Histopathology Image Classification using Machine Learning

This notebook documents a portfolio-safe machine learning workflow for classifying colon cell histopathology image patches.

The original coursework version used a modified CRCHistoPhenotypes-style dataset of 27×27 RGB image patches. The dataset, label files, images, reports, slides, and submission files are not included in this repository due to coursework data-use restrictions.

## Project tasks

1. **Cancer classification:** predict whether a cell image is cancerous or non-cancerous.
2. **Cell-type classification:** predict the cell type as fibroblast, inflammatory, epithelial, or others.

## Main workflow

- dataset inspection and quality checks  
- patient-level train/validation/test splitting  
- image preprocessing and normalisation  
- PCA-based exploratory visualisation  
- classical machine learning baselines  
- CNN modelling with checkpointing  
- data augmentation and ensemble ideas  
- threshold tuning for binary classification  
- final evaluation using accuracy, macro-F1, weighted-F1, classification reports, and confusion matrices  

This version is designed for GitHub portfolio presentation and excludes all restricted data.

## 1. Imports and configuration

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

## 2. Expected project structure

This notebook expects the following local structure if the dataset is available privately:

```text
project/
├── histopathology_image_classification.ipynb
└── data/
    ├── images/
    ├── data_labels_mainData.csv
    └── data_labels_extraData.csv
```

The `data/` folder is intentionally excluded from GitHub.

In [ ]:
DATA_DIR = Path("data")
IMAGE_DIR = DATA_DIR / "images"
MAIN_LABELS_PATH = DATA_DIR / "data_labels_mainData.csv"
EXTRA_LABELS_PATH = DATA_DIR / "data_labels_extraData.csv"

print("Main labels file exists:", MAIN_LABELS_PATH.exists())
print("Extra labels file exists:", EXTRA_LABELS_PATH.exists())
print("Image folder exists:", IMAGE_DIR.exists())

## 3. Load label files

The main label file is used for both tasks because it contains both `isCancerous` and `cellType` labels.  
The extra label file is used only for the cancer classification task because it contains `isCancerous` labels only.

In [ ]:
if MAIN_LABELS_PATH.exists() and EXTRA_LABELS_PATH.exists():
    main_df = pd.read_csv(MAIN_LABELS_PATH)
    extra_df = pd.read_csv(EXTRA_LABELS_PATH)

    print("Main labelled records:", len(main_df))
    print("Extra cancer-labelled records:", len(extra_df))
    print("Main patients:", main_df["patientID"].nunique())
    print("Extra patients:", extra_df["patientID"].nunique())
else:
    raise FileNotFoundError(
        "Dataset files are not included in this repository. "
        "Place the private coursework data inside the data/ folder to run this notebook."
    )

## 4. Basic data quality checks

This step checks missing values, duplicate image names, patient counts, and target distributions.  
Only summary statistics are printed; raw rows are not displayed in this GitHub-safe version.

In [ ]:
def summarise_label_file(df, name):
    print(f"\n{name}")
    print("-" * len(name))
    print("Rows:", len(df))
    print("Columns:", list(df.columns))
    print("Patients:", df["patientID"].nunique())
    print("Missing values:", int(df.isna().sum().sum()))
    print("Duplicate image names:", int(df["ImageName"].duplicated().sum()))

summarise_label_file(main_df, "Main label file")
summarise_label_file(extra_df, "Extra label file")

print("\nCancer label distribution: main")
print(main_df["isCancerous"].value_counts().sort_index())

print("\nCancer label distribution: extra")
print(extra_df["isCancerous"].value_counts().sort_index())

print("\nCell-type distribution: main")
print(main_df["cellTypeName"].value_counts())

## 5. Image availability check

The original workflow verified that all images referenced in the label files were present in the image folder.  
This is important because missing images can silently reduce the dataset and affect reproducibility.

In [ ]:
def check_image_availability(label_df, image_dir, image_col="ImageName"):
    available_images = set(os.listdir(image_dir))
    listed_images = set(label_df[image_col])
    missing_images = listed_images - available_images
    return len(listed_images), len(missing_images)

main_count, main_missing = check_image_availability(main_df, IMAGE_DIR)
extra_count, extra_missing = check_image_availability(extra_df, IMAGE_DIR)

print("Main images listed:", main_count, "| Missing:", main_missing)
print("Extra images listed:", extra_count, "| Missing:", extra_missing)

## 6. Patient-level train/validation/test split

The dataset is split by `patientID`, not by individual image.  
This avoids data leakage where images from the same patient appear in both training and testing sets.

This is stricter and more realistic for medical imaging because the final test set represents unseen patients.

In [ ]:
def patient_level_split(df, patient_col="patientID", test_size=0.20, val_size=0.20, random_state=SEED):
    patients = df[patient_col].unique()

    train_val_patients, test_patients = train_test_split(
        patients,
        test_size=test_size,
        random_state=random_state,
    )

    train_patients, val_patients = train_test_split(
        train_val_patients,
        test_size=val_size,
        random_state=random_state,
    )

    train_df = df[df[patient_col].isin(train_patients)].reset_index(drop=True)
    val_df = df[df[patient_col].isin(val_patients)].reset_index(drop=True)
    test_df = df[df[patient_col].isin(test_patients)].reset_index(drop=True)

    return train_df, val_df, test_df, train_patients, val_patients, test_patients


cancer_df = pd.concat(
    [
        main_df[["InstanceID", "patientID", "ImageName", "isCancerous"]].copy(),
        extra_df[["InstanceID", "patientID", "ImageName", "isCancerous"]].copy(),
    ],
    ignore_index=True,
)

celltype_df = main_df[
    ["InstanceID", "patientID", "ImageName", "cellType", "cellTypeName", "isCancerous"]
].copy()

cancer_train_df, cancer_val_df, cancer_test_df, cancer_train_patients, cancer_val_patients, cancer_test_patients = patient_level_split(cancer_df)
cell_train_df, cell_val_df, cell_test_df, cell_train_patients, cell_val_patients, cell_test_patients = patient_level_split(celltype_df)

print("Cancer split:", cancer_train_df.shape, cancer_val_df.shape, cancer_test_df.shape)
print("Cell-type split:", cell_train_df.shape, cell_val_df.shape, cell_test_df.shape)

In [ ]:
def check_patient_overlap(train_patients, val_patients, test_patients):
    train_set = set(train_patients)
    val_set = set(val_patients)
    test_set = set(test_patients)

    return {
        "train_validation_overlap": len(train_set & val_set),
        "train_test_overlap": len(train_set & test_set),
        "validation_test_overlap": len(val_set & test_set),
    }

print("Cancer patient overlap:", check_patient_overlap(cancer_train_patients, cancer_val_patients, cancer_test_patients))
print("Cell-type patient overlap:", check_patient_overlap(cell_train_patients, cell_val_patients, cell_test_patients))

## 7. Image loading and preprocessing

Images are loaded as RGB arrays and normalised from 0–255 to 0–1.

Two feature formats are prepared:

1. **Flattened pixel vectors** for classical ML models.  
2. **RGB image tensors** for CNN models.

In [ ]:
def load_images_as_arrays(df, image_dir=IMAGE_DIR, image_col="ImageName"):
    images = []

    for image_name in df[image_col]:
        image_path = image_dir / image_name
        image = Image.open(image_path).convert("RGB")
        image_array = np.asarray(image, dtype=np.float32) / 255.0
        images.append(image_array)

    return np.stack(images)


def flatten_images(image_array):
    return image_array.reshape(image_array.shape[0], -1)

In [ ]:
# Load images for both tasks.
# This can take some time depending on hardware.

X_cancer_train_img = load_images_as_arrays(cancer_train_df)
X_cancer_val_img = load_images_as_arrays(cancer_val_df)
X_cancer_test_img = load_images_as_arrays(cancer_test_df)

y_cancer_train = cancer_train_df["isCancerous"].values
y_cancer_val = cancer_val_df["isCancerous"].values
y_cancer_test = cancer_test_df["isCancerous"].values

X_cancer_train_flat = flatten_images(X_cancer_train_img)
X_cancer_val_flat = flatten_images(X_cancer_val_img)
X_cancer_test_flat = flatten_images(X_cancer_test_img)

X_cell_train_img = load_images_as_arrays(cell_train_df)
X_cell_val_img = load_images_as_arrays(cell_val_df)
X_cell_test_img = load_images_as_arrays(cell_test_df)

y_cell_train = cell_train_df["cellType"].values
y_cell_val = cell_val_df["cellType"].values
y_cell_test = cell_test_df["cellType"].values

X_cell_train_flat = flatten_images(X_cell_train_img)
X_cell_val_flat = flatten_images(X_cell_val_img)
X_cell_test_flat = flatten_images(X_cell_test_img)

print("Cancer image array:", X_cancer_train_img.shape)
print("Cell-type image array:", X_cell_train_img.shape)
print("Flattened feature size:", X_cancer_train_flat.shape[1])

## 8. PCA exploratory visualisation

PCA is used only for exploratory visualisation of flattened image features.  
The goal is to check whether classes separate clearly in a low-dimensional projection.

In [ ]:
def plot_pca_projection(X, y, title, sample_size=3000):
    if len(X) > sample_size:
        idx = np.random.choice(len(X), sample_size, replace=False)
        X_sample = X[idx]
        y_sample = y[idx]
    else:
        X_sample = X
        y_sample = y

    pca = PCA(n_components=2, random_state=SEED)
    X_pca = pca.fit_transform(X_sample)

    plt.figure(figsize=(7, 5))
    scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_sample, alpha=0.6, s=12)
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.title(title)
    plt.colorbar(scatter)
    plt.tight_layout()
    plt.show()

    print("Explained variance ratio:", pca.explained_variance_ratio_)
    print("Total explained variance:", pca.explained_variance_ratio_.sum())


plot_pca_projection(X_cancer_train_flat, y_cancer_train, "PCA projection: cancer labels")
plot_pca_projection(X_cell_train_flat, y_cell_train, "PCA projection: cell-type labels")

## 9. Classical machine learning baselines

Classical models are trained using flattened pixel features.  
These models provide useful baselines before moving to CNNs.

In [ ]:
def evaluate_classifier(model, X_train, y_train, X_val, y_val, model_name):
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)

    result = {
        "Model": model_name,
        "Train Accuracy": accuracy_score(y_train, train_pred),
        "Validation Accuracy": accuracy_score(y_val, val_pred),
        "Train Macro F1": f1_score(y_train, train_pred, average="macro"),
        "Validation Macro F1": f1_score(y_val, val_pred, average="macro"),
        "Validation Weighted F1": f1_score(y_val, val_pred, average="weighted"),
    }

    return result, val_pred


cancer_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
    ]),
    "Linear SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearSVC(class_weight="balanced", max_iter=5000))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=SEED,
        class_weight="balanced",
        n_jobs=-1,
    ),
}

cancer_results = []

for name, model in cancer_models.items():
    result, _ = evaluate_classifier(
        model,
        X_cancer_train_flat,
        y_cancer_train,
        X_cancer_val_flat,
        y_cancer_val,
        name,
    )
    cancer_results.append(result)

cancer_results_df = pd.DataFrame(cancer_results)
cancer_results_df

In [ ]:
cell_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
    ]),
    "Linear SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearSVC(class_weight="balanced", max_iter=5000))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=SEED,
        class_weight="balanced",
        n_jobs=-1,
    ),
}

cell_results = []

for name, model in cell_models.items():
    result, _ = evaluate_classifier(
        model,
        X_cell_train_flat,
        y_cell_train,
        X_cell_val_flat,
        y_cell_val,
        name,
    )
    cell_results.append(result)

cell_results_df = pd.DataFrame(cell_results)
cell_results_df

## 10. Random Forest tuning example

The original workflow tuned model complexity using validation macro-F1.  
The example below shows how `max_depth` can be tuned for the cancer classification task.

In [ ]:
depth_values = [5, 10, 15, 20, None]
rf_tuning_results = []

for depth in depth_values:
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=depth,
        random_state=SEED,
        class_weight="balanced",
        n_jobs=-1,
    )

    model.fit(X_cancer_train_flat, y_cancer_train)
    val_pred = model.predict(X_cancer_val_flat)

    rf_tuning_results.append({
        "max_depth": str(depth),
        "validation_accuracy": accuracy_score(y_cancer_val, val_pred),
        "validation_macro_f1": f1_score(y_cancer_val, val_pred, average="macro"),
    })

rf_tuning_df = pd.DataFrame(rf_tuning_results)
rf_tuning_df

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(rf_tuning_df["max_depth"], rf_tuning_df["validation_macro_f1"], marker="o")
plt.xlabel("max_depth")
plt.ylabel("Validation macro-F1")
plt.title("Random Forest depth tuning")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. PyTorch dataset and CNN model

CNNs preserve local spatial information, which is important for image classification tasks.  
The simple CNN below is intentionally compact because the image patches are only 27×27 pixels.

In [ ]:
class HistopathologyDataset(Dataset):
    def __init__(self, image_array, labels, transform=None):
        self.image_array = image_array
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_array)

    def __getitem__(self, idx):
        image = self.image_array[idx]
        label = int(self.labels[idx])

        image = Image.fromarray((image * 255).astype(np.uint8))

        if self.transform is not None:
            image = self.transform(image)
        else:
            image = transforms.ToTensor()(image)

        return image, label


class SmallCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

In [ ]:
def train_cnn(model, train_loader, val_loader, epochs=10, learning_rate=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    history = []
    best_macro_f1 = -1
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []

        for images, labels in train_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        model.eval()
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(DEVICE)
                outputs = model(images)
                preds = torch.argmax(outputs, dim=1).cpu().numpy()

                all_preds.extend(preds)
                all_labels.extend(labels.numpy())

        val_acc = accuracy_score(all_labels, all_preds)
        val_macro_f1 = f1_score(all_labels, all_preds, average="macro")

        history.append({
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            "val_accuracy": val_acc,
            "val_macro_f1": val_macro_f1,
        })

        if val_macro_f1 > best_macro_f1:
            best_macro_f1 = val_macro_f1
            best_state = model.state_dict()

        print(
            f"Epoch {epoch:02d} | "
            f"Loss: {np.mean(train_losses):.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val Macro-F1: {val_macro_f1:.4f}"
        )

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

## 12. CNN training example

The validation set is used for checkpointing.  
The best epoch is selected by validation macro-F1 rather than the final epoch.

In [ ]:
basic_transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = HistopathologyDataset(X_cell_train_img, y_cell_train, transform=basic_transform)
val_dataset = HistopathologyDataset(X_cell_val_img, y_cell_val, transform=basic_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

cell_cnn = SmallCNN(num_classes=4).to(DEVICE)
cell_cnn, cell_cnn_history = train_cnn(
    cell_cnn,
    train_loader,
    val_loader,
    epochs=10,
    learning_rate=1e-3,
)

cell_cnn_history

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(cell_cnn_history["epoch"], cell_cnn_history["train_loss"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("CNN training loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(cell_cnn_history["epoch"], cell_cnn_history["val_macro_f1"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Validation macro-F1")
plt.title("CNN validation macro-F1")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 13. Data augmentation example

Data augmentation was investigated as an advanced technique.  
Augmentation is applied only to the training data, not to validation or test data.

In [ ]:
augmentation_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.10),
    transforms.ToTensor(),
])

augmented_train_dataset = HistopathologyDataset(
    X_cancer_train_img,
    y_cancer_train,
    transform=augmentation_transform,
)

cancer_val_dataset = HistopathologyDataset(
    X_cancer_val_img,
    y_cancer_val,
    transform=basic_transform,
)

augmented_train_loader = DataLoader(augmented_train_dataset, batch_size=64, shuffle=True)
cancer_val_loader = DataLoader(cancer_val_dataset, batch_size=128, shuffle=False)

cancer_aug_cnn = SmallCNN(num_classes=2).to(DEVICE)
cancer_aug_cnn, cancer_aug_history = train_cnn(
    cancer_aug_cnn,
    augmented_train_loader,
    cancer_val_loader,
    epochs=10,
    learning_rate=1e-3,
)

## 14. Binary threshold tuning

For binary cancer classification, threshold tuning can be used when the model outputs probabilities.  
The best threshold is chosen using validation macro-F1.

In [ ]:
def find_best_threshold(y_true, positive_probabilities, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.1, 0.9, 81)

    records = []

    for threshold in thresholds:
        preds = (positive_probabilities >= threshold).astype(int)

        records.append({
            "threshold": threshold,
            "macro_f1": f1_score(y_true, preds, average="macro"),
            "accuracy": accuracy_score(y_true, preds),
        })

    result_df = pd.DataFrame(records)
    best_row = result_df.loc[result_df["macro_f1"].idxmax()]
    return result_df, best_row


# Example usage:
# probabilities = model.predict_proba(X_cancer_val_flat)[:, 1]
# threshold_results, best_threshold = find_best_threshold(y_cancer_val, probabilities)
# best_threshold

## 15. Final project summary

The original coursework experiments selected:

- **Cancer classification:** tuned Random Forest + augmented CNN soft-voting ensemble with threshold tuning.
- **Cell-type classification:** CNN with checkpointing.

The final models were evaluated once on unseen patients after validation-based model selection.

## Responsible interpretation

This project should be interpreted as an exploratory machine learning workflow, not as a clinical diagnostic system.

Important limitations include:

- the dataset was a modified coursework dataset;
- the images were very small 27×27 local patches;
- performance was evaluated internally rather than on an external clinical cohort;
- false negatives in cancer classification would be clinically serious;
- the `others` cell-type class remained difficult and visually heterogeneous.

A real deployment would require expert pathologist review, external validation, uncertainty estimation, bias monitoring, explainability, governance, and integration with broader clinical context.

## 16. Data-use note

The dataset and label files are not included in this repository.  
This notebook is provided as a portfolio-safe summary of the workflow and learning outcomes without redistributing restricted coursework files.